In [ ]:
import re
import sys
import numpy as np
from pathlib import Path as ph

In [ ]:
# Add parent directory to sys.path
sys.path.append(str(ph().resolve().parent))
from src.functions.runtime import from_file, towa_file, get_directory_files

In [ ]:
runtime_path_inp = input("Enter the runtime path ('same','<path>'): ").strip().lower()
runtime_uuid_inp = input("Enter the model configuration ('<uuid>','all'): ").strip().lower()

In [ ]:
########################
# Runtime variables
########################

if runtime_path_inp == "same":
    runtime_path = "."
else:
    runtime_path = runtime_path_inp

runtime_uuid = runtime_uuid_inp

configs_path = f"{runtime_path}/configs"
tokenizers_path = f"{runtime_path}/tokenizers"
inputs_path = f"{runtime_path}/inputs"
outputs_path = f"{runtime_path}/outputs"
models_path = f"{runtime_path}/models"
charts_path = f"{runtime_path}/charts"
statistics_path = f"{runtime_path}/statistics"

In [ ]:
########################
# Add row experiment file components
########################

def process_add_row_expirement_file_components(folder_path, file_prefix, runtime_uuid):
    runtime_uuids = []
    if runtime_uuid == "all":
        # If a all is provided, add ithem all to the list
        runtime_uuids = get_directory_files(folder_path, file_prefix)
    else:
        # If a specific UUID is provided, add it to the list
        runtime_uuids.append(runtime_uuid)

    statistic_path = f"{statistics_path}/statistic_experiments.csv"
    statistic = from_file(statistic_path, "csv")

    # Loop through each GUID to extract weights
    for runtime_uuid in runtime_uuids:
        print(runtime_uuid)
        # Load the model from a single file
        config_path_inp = f"{configs_path}/config_{runtime_uuid}.json"
        config_json = from_file(config_path_inp, "json")
        report_path_inp = f"{outputs_path}/report_{runtime_uuid}.json"
        report_json = from_file(report_path_inp, "json")
        completion_path_inp = f"{outputs_path}/completion_{runtime_uuid}.json"
        completion_json = from_file(completion_path_inp, "json")

        row = [
            runtime_uuid,
            config_json["runtime"]["model_version"],
            config_json["runtime"]["model_params"],
            config_json["runtime"]["model_size"],
            config_json["runtime"]["device_name"],
            config_json["runtime"]["library_driver"],
            config_json["c_device"],
            config_json["c_tokenizer"],
            config_json["c_sequence"],
            config_json["c_attention"],
            config_json["c_network"],
            config_json["n_ctx"],
            config_json["n_emb"],
            config_json["r_dropout"],
            config_json["s_head"],
            config_json["n_heads"],
            config_json["n_layers"],
            config_json["n_epochs"],
            config_json["s_batch"],
            config_json["r_learn"],
            report_json["dataset"],
            f"{report_json["avg_train_time_per_batch"]:.2f}",
            f"{report_json["avg_val_time_per_batch"]:.2f}",
            f"{report_json["average_time_per_epoch"]:.0f}",
            f"{report_json["total_time"]:.0f}",
            f"{completion_json["inference_total_time"]:.0f}",
            f"{completion_json["evaluation"]:.1f}/100"
        ]
        print(f"{'-'*10} {'Adding Row'} {'-'*10}")
        print(row)
        statistic.append(row)

        # Reconstruct the statistic with header and added data
        towa_file(statistic_path, "csv", statistic)

In [ ]:
# === Run the script ===
folder_path = f"{runtime_path}/configs"
file_prefix = f"config"
process_add_row_expirement_file_components(folder_path, file_prefix, runtime_uuid)